# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atulpatel-net/FlyRank_ML_In/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents the daily search and analytics performance of a single content item for a single client on a specific reporting date.

### Time window

This notebook uses data from the `fact_content_daily_performance` table for the month `2026-03`. The goal is to use historical information available at the decision time to rank content pages for refresh priority.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
These fields are available before the prediction/decision point and represent observed search and engagement signals.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- sessions_organic
- scroll_events

### Label / Proxy
The objective is to rank content for refresh opportunity using historical search and engagement signals. The ranking score (or refresh priority) is treated as the target/proxy and is not used as an input feature.

### Context
These fields identify or organize the data but are not model inputs.

- report_date
- client_hash_id
- content_hash_id
- month

### Excluded
These fields are excluded because they are not predictive signals or could introduce bias/leakage.

- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS duplicate_rows
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

,duplicate_rows
0,0


In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS usable_rows
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

,usable_rows
0,364347


In [19]:
con.sql(f"""
SELECT
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS gsc_impressions_nulls,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS gsc_clicks_nulls,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS gsc_avg_position_nulls,
    SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS ga4_pageviews_nulls,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS ga4_sessions_nulls
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

,gsc_impressions_nulls,gsc_clicks_nulls,gsc_avg_position_nulls,ga4_pageviews_nulls,ga4_sessions_nulls
0,0.0,0.0,0.0,0.0,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

- This dataset contains historical search and analytics metrics, but it cannot explain *why* performance changed (for example, algorithm updates, seasonality, competitor actions, or content quality changes).

- Early periods may contain rows where only GSC data is available, reducing the number of records that can use both GSC and GA4 metrics together.

- Using a single month (March 2026) provides only a snapshot of performance and may not capture long-term trends or seasonal behavior.

- Care must be taken to avoid overlapping observation and outcome windows, as using future information would introduce data leakage into any predictive model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.